# 01 — English CLIP Semantic Closeness (within categories)

Takes the deduped THINGSplus candidates and, within each **category** (`primary_category`,
derived from THINGSplus's `cat53_string`), selects the N semantically **closest** words —
i.e. the tightest mutual cluster.

**Uncategorized words:** THINGSplus leaves `cat53_string` blank for some concepts. These are
excluded from category grouping by default — output: `uncategorized_words.csv`.

**Multi-label categories:** THINGSplus sometimes gives multiple category labels per concept
(e.g. "food; fruit"). This uses the **first** listed label as the primary category.


In [1]:
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS


/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ===== Config =====
CSV_PATH = "output/unique_candidates_english_zipf.csv"
WORD_COL = "word_lower"
GROUP_COL = "primary_category"
DEVICE = "cpu"                 # or "cuda"
SAVE_DIR = "output/mds_plots_en_category"
ZIPF_COL = "ZipfSUBTLEX_en"
N_SELECT = 3                   # how many closest words to pull per category


In [ ]:
# May need more words to exclude
EXCLUDE_WORDS = ["can"]

df = pd.read_csv(CSV_PATH)
df["word_lower"] = df["Word"].str.lower()
df = df[~df[WORD_COL].isin(EXCLUDE_WORDS)].reset_index(drop=True)

# Derive primary category from cat53_string (first label before ';')
df["primary_category"] = df["cat53_string"].str.split(";").str[0].str.strip()

uncategorized = df[df["primary_category"].isna()]
uncategorized.to_csv("output/uncategorized_words.csv", index=False)
print(f"{len(uncategorized)} uncategorized words saved to output/uncategorized_words.csv (excluded below)")

df = df.dropna(subset=["primary_category"]).reset_index(drop=True)
groups = {g: grp[WORD_COL].tolist() for g, grp in df.groupby(GROUP_COL)}
print({g: len(v) for g, v in sorted(groups.items(), key=lambda x: -len(x[1]))})


98 uncategorized words saved to output/uncategorized_words.csv (excluded below)
{'food': 73, 'clothing': 35, 'container': 26, 'arts and crafts supply': 21, 'electronic device': 18, 'sports equipment': 17, 'musical instrument': 16, 'personal hygiene item': 16, 'home decor': 14, 'body part': 12, 'hardware': 12, 'kitchen tool': 11, 'clothing accessory': 7, 'office supply': 7, 'part of car': 7, 'medical equipment': 6, 'scientific equipment': 6, 'tool': 6, 'toy': 6, 'animal': 5, 'breakfast food': 5, 'construction equipment': 5, 'game': 5, 'condiment': 4, 'fastener': 4, 'jewelry': 3, 'plant': 3, 'safety equipment': 3, 'school supply': 3, 'candy': 2, 'dessert': 2, 'drink': 2, 'headwear': 2, 'lighting': 2, 'footwear': 1, 'furniture': 1, 'garden tool': 1, 'protective clothing': 1}


In [ ]:
# ===== Load CLIP model =====
# Using the same multilingual CLIP model as the German pipeline, so English and German
# embeddings live in the same space and results are directly comparable across languages.
model = SentenceTransformer("sentence-transformers/clip-ViT-B-32-multilingual-v1", device=DEVICE)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1864.25it/s]


In [5]:
# Encode all unique words once
all_words = sorted(set(df[WORD_COL]))
emb_all = model.encode(all_words, normalize_embeddings=True)
word2vec = {w: emb_all[i] for i, w in enumerate(all_words)}


In [ ]:
# ===== Find N most mutually CLOSE words within a category =====
def find_most_close_set(words, D, n=3):
    best_sum = np.inf
    best_combo = None
    for combo in itertools.combinations(range(len(words)), n):
        first_letters = {words[i][0].lower() for i in combo}
        if len(first_letters) < n:
            continue  # skip combos with a repeated first letter
        total_dist = sum(D[i, j] for i, j in itertools.combinations(combo, 2))
        if total_dist < best_sum:
            best_sum = total_dist
            best_combo = combo
    if best_combo is None:
        # fall back to ignoring the first-letter constraint if nothing satisfies it
        for combo in itertools.combinations(range(len(words)), n):
            total_dist = sum(D[i, j] for i, j in itertools.combinations(combo, 2))
            if total_dist < best_sum:
                best_sum = total_dist
                best_combo = combo
    return [words[i] for i in best_combo], best_sum


In [7]:
# ===== Plot MDS per category =====
def plot_group(words, emb, group_id, selected_words, mean_dist, save_dir=""):
    if len(words) < 2:
        print(f"Category '{group_id}' has too few words, skipping plot")
        return
    D = pairwise_distances(emb, metric="cosine")
    coords = MDS(n_components=2, dissimilarity="precomputed", random_state=42, n_init=4).fit_transform(D)
    sel_set = set(selected_words)

    plt.figure(figsize=(6, 5))
    idx_other = [i for i, w in enumerate(words) if w not in sel_set]
    plt.scatter(coords[idx_other, 0], coords[idx_other, 1], c="#9aa0a6", s=30, label="other words")
    idx_sel = [i for i, w in enumerate(words) if w in sel_set]
    plt.scatter(coords[idx_sel, 0], coords[idx_sel, 1], c="#1a73e8", s=60, label="closest words")
    for i in idx_sel:
        plt.annotate(words[i], (coords[i, 0], coords[i, 1]),
                     xytext=(4, 4), textcoords="offset points", fontsize=9, color="#1a73e8")

    plt.title(f"{group_id} (EN) — MDS\nAvg_Distance={mean_dist:.3f}")
    plt.xlabel("MDS-1"); plt.ylabel("MDS-2")
    plt.legend(frameon=False); plt.grid(alpha=0.2)
    plt.tight_layout()

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        safe_name = str(group_id).replace(" ", "_").replace("/", "-")
        plt.savefig(os.path.join(save_dir, f"{safe_name}_mds_en.png"), dpi=150)
        plt.close()
    else:
        plt.show()


In [8]:
# ===== Main loop =====
results = []
skipped = []
for g in sorted(groups.keys()):
    words = groups[g]
    if len(words) < N_SELECT:
        skipped.append((g, len(words)))
        continue
    emb = np.stack([word2vec[w] for w in words])
    D = pairwise_distances(emb, metric="cosine")
    selected_words, total_dist = find_most_close_set(words, D, n=N_SELECT)
    mean_dist = total_dist / (N_SELECT * (N_SELECT - 1) / 2)
    plot_group(words, emb, g, selected_words, mean_dist, save_dir=SAVE_DIR)

    sel_df = df[df[WORD_COL].isin(selected_words) & (df["primary_category"] == g)][
        [WORD_COL, "Word", ZIPF_COL, "primary_category"]
    ]
    for _, r in sel_df.iterrows():
        results.append({
            "category": g,
            "word": r[WORD_COL],
            "Word_original": r["Word"],
            ZIPF_COL: r[ZIPF_COL],
            "category_total_distance": total_dist,
            "category_mean_distance": mean_dist
        })

res_df = pd.DataFrame(results)
print(res_df)
if skipped:
    print(f"\nSkipped {len(skipped)} categories with fewer than {N_SELECT} candidate words:")
    for g, n in skipped:
        print(f"  {g}: {n} words")


/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` instead.
  warnings.warn(
/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:735: FutureWarning: The default value of `init` will change from 'random' to 'classical_mds' in 1.10. To suppress this warning, provide some value of `init`.
  warnings.warn(
/home/hivrim8h/projects/EventSegmentation/.venv/lib/python3.12/site-packages/sklearn/manifold/_mds.py:752: FutureWarning: The `dissimilarity` parameter is deprecated and will be removed in 1.10. Use `metric` inst

                  category      word Word_original  ZipfSUBTLEX_en  \
0                   animal   chicken       chicken        4.790465   
1                   animal   hamster       hamster        3.329856   
2                   animal     snail         snail        3.246672   
3   arts and crafts supply     paper         paper        5.014323   
4   arts and crafts supply    marker        marker        3.718941   
..                     ...       ...           ...             ...   
82                    tool    duster        duster        2.593460   
83                    tool  squeegee      squeegee        2.438558   
84                     toy      doll          doll        4.393833   
85                     toy       toy           toy        4.226423   
86                     toy      kite          kite        3.360616   

    category_total_distance  category_mean_distance  
0                  0.482408                0.160803  
1                  0.482408                0.160803

In [9]:
# ===== Export =====
res_df.to_csv("output/closest_selection_english.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(res_df)} selected words across {res_df['category'].nunique()} categories "
      f"to output/closest_selection_english.csv")


Saved 87 selected words across 29 categories to output/closest_selection_english.csv
